# Test hmm_soft_residual por K (cosine LR)
Ejecuta hmm_soft_residual para cada K cacheado. Cada celda = 1 K. Si ya existe el checkpoint, skip.

In [1]:
import os, subprocess, glob

# Ejecutar desde raiz del proyecto
os.chdir('/home/jaime/TFG/RITMO')

# Detectar Ks disponibles
K_values = sorted([int(f.split('_K')[1].split('.')[0]) for f in glob.glob('./cache/hmm_etth1_K*.pth')])
print(f'Ks disponibles: {K_values}')

TECHNIQUE = 'hmm_soft_residual'

# Params comunes
BASE_CMD = [
    'python', '-u', 'run.py',
    '--task_name', 'plan_a',
    '--is_training', '1',
    '--root_path', './dataset/ETT-small/',
    '--data_path', 'ETTh1.csv',
    '--model_id', 'ETTh1_96_96',
    '--model', 'TransformerCommon',
    '--data', 'ETTh1',
    '--features', 'S',
    '--seq_len', '96',
    '--pred_len', '96',
    '--enc_in', '1',
    '--dec_in', '1',
    '--c_out', '1',
    '--d_model', '256',
    '--n_heads', '4',
    '--e_layers', '3',
    '--d_ff', '512',
    '--dropout', '0.1',
    '--batch_size', '32',
    '--learning_rate', '0.0001',
    '--lradj', 'cosine',
    '--train_epochs', '20',
    '--patience', '5',
    '--use_gpu', '0',
    '--technique', TECHNIQUE,
    '--itr', '1',
]

def setting_for_K(K):
    des = f'Plan_A_{TECHNIQUE}_K{K}'
    return f'plan_a_ETTh1_96_96_TransformerCommon_ETTh1_ftS_sl96_ll48_pl96_dm256_nh4_el3_dl1_df512_expand2_dc4_fc1_ebtimeF_dtTrue_{des}_0'

def run_K(K):
    ckpt = f'./checkpoints/{setting_for_K(K)}/checkpoint.pth'
    if os.path.exists(ckpt):
        print(f'K={K}: SKIP - checkpoint ya existe')
        return
    des = f'Plan_A_{TECHNIQUE}_K{K}'
    cmd = BASE_CMD + ['--hmm_k', str(K), '--des', des]
    print(f'K={K}: ejecutando...')
    r = subprocess.run(cmd, capture_output=False)
    print(f'K={K}: {"OK" if r.returncode == 0 else "ERROR"}')

print(f'Tecnica: {TECHNIQUE}')
print('Funcion run_K(K) lista. Ejecuta cada celda de abajo.')

Ks disponibles: [5, 10, 15, 20]
Tecnica: hmm_soft_residual
Funcion run_K(K) lista. Ejecuta cada celda de abajo.


In [2]:
run_K(5)

K=5: ejecutando...
Using cpu or mps
Args in experiment:
Basic Config
  Task Name:          plan_a              Is Training:        1                   
  Model ID:           ETTh1_96_96         Model:              TransformerCommon   

Data Loader
  Data:               ETTh1               Root Path:          ./dataset/ETT-small/
  Data Path:          ETTh1.csv           Features:           S                   
  Target:             OT                  Freq:               h                   
  Checkpoints:        ./checkpoints/      

Model Parameters
  Top k:              5                   Num Kernels:        6                   
  Enc In:             1                   Dec In:             1                   
  C Out:              1                   d model:            256                 
  n heads:            4                   e layers:           3                   
  d layers:           1                   d FF:               512                 
  Moving Avg:         25   

In [ ]:
run_K(10)

K=10: ejecutando...
Using cpu or mps
Args in experiment:
Basic Config
  Task Name:          plan_a              Is Training:        1                   
  Model ID:           ETTh1_96_96         Model:              TransformerCommon   

Data Loader
  Data:               ETTh1               Root Path:          ./dataset/ETT-small/
  Data Path:          ETTh1.csv           Features:           S                   
  Target:             OT                  Freq:               h                   
  Checkpoints:        ./checkpoints/      

Model Parameters
  Top k:              5                   Num Kernels:        6                   
  Enc In:             1                   Dec In:             1                   
  C Out:              1                   d model:            256                 
  n heads:            4                   e layers:           3                   
  d layers:           1                   d FF:               512                 
  Moving Avg:         25  

In [ ]:
run_K(15)

In [ ]:
run_K(20)

In [ ]:
# Resumen de resultados
import numpy as np

print(f'{"K":>3} | {"MSE":>10} | {"MAE":>10}')
print('-' * 32)
for K in K_values:
    metrics_path = f'../results/{setting_for_K(K)}/metrics.npy'
    if os.path.exists(metrics_path):
        mae, mse, rmse, mape, mspe = np.load(metrics_path)
        print(f'{K:3d} | {mse:10.6f} | {mae:10.6f}')
    else:
        print(f'{K:3d} | {"pendiente":>10} | {"pendiente":>10}')